In [0]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [0]:
spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")
spark.conf.set("spark.sql.shuffle.partitions", "200")
spark.conf.set("spark.sql.adaptive.enabled", "true")

In [0]:
from config import PATHS, TABLES, MONITORING_TABLE

In [0]:
spark.sql("DROP TABLE IF EXISTS dbw_retails.gold.customer_revenue")
spark.sql("DROP TABLE IF EXISTS dbw_retails.gold.product_performance")
spark.sql("DROP TABLE IF EXISTS dbw_retails.gold.country_sales")

spark.sql("DROP TABLE IF EXISTS dbw_retails.silver.online_retail_transactions")

spark.sql("DROP TABLE IF EXISTS dbw_retails.bronze.eventhub_transactions")
spark.sql("DROP TABLE IF EXISTS dbw_retails.bronze.file_transactions")

spark.sql("DROP TABLE IF EXISTS dbw_retails.monitoring.pipeline_metrics")

DataFrame[]

In [0]:
dbutils.fs.rm(PATHS["bronze_eventhub"], True)
dbutils.fs.rm(PATHS["bronze_files"], True)

dbutils.fs.rm(PATHS["silver"], True)

dbutils.fs.rm(PATHS["gold_customer"], True)
dbutils.fs.rm(PATHS["gold_product"], True)
dbutils.fs.rm(PATHS["gold_country"], True)

dbutils.fs.rm(PATHS["monitoring"], True)

True

In [0]:
from config import CHECKPOINTS

dbutils.fs.rm(CHECKPOINTS["bronze_eventhub"], True)
dbutils.fs.rm(CHECKPOINTS["bronze_files"], True)
dbutils.fs.rm(CHECKPOINTS["silver"], True)

False

In [0]:
from config import CHECKPOINTS

dbutils.fs.rm(CHECKPOINTS["bronze_eventhub"], True)
dbutils.fs.rm(CHECKPOINTS["bronze_files"], True)
dbutils.fs.rm(CHECKPOINTS["silver"], True)

False

In [0]:
spark.sql(f"""
CREATE TABLE {TABLES["bronze_eventhub"]} (
  InvoiceNo STRING,
  StockCode STRING,
  Description STRING,
  Quantity INT,
  InvoiceDate STRING,
  UnitPrice DOUBLE,
  CustomerID DOUBLE,
  Country STRING,
  _ingest_ts TIMESTAMP
)
USING DELTA
LOCATION '{PATHS["bronze_eventhub"]}'
""")

DataFrame[]

In [0]:
spark.sql(f"""
CREATE TABLE {TABLES["bronze_files"]} (
  InvoiceNo STRING,
  StockCode STRING,
  Description STRING,
  Quantity INT,
  InvoiceDate STRING,
  UnitPrice DOUBLE,
  CustomerID DOUBLE,
  Country STRING,
  _ingest_ts TIMESTAMP,
  _source_file STRING
)
USING DELTA
LOCATION '{PATHS["bronze_files"]}'
""")

DataFrame[]

In [0]:
from config import PATHS

spark.sql(f"""
CREATE TABLE IF NOT EXISTS dbw_retails.silver.online_retail_transactions (
    invoice_no STRING,
    stock_code STRING,
    description STRING,
    quantity INT,
    invoice_date STRING,
    unit_price DOUBLE,
    customer_id DOUBLE,
    country STRING,
    invoice_ts TIMESTAMP,
    invoice_dt DATE,
    _ingest_ts TIMESTAMP,
    _source_file STRING,
    revenue DOUBLE,
    year INT,
    month INT,
    day INT,
    pipeline_stage STRING
)
USING DELTA
LOCATION '{PATHS["silver"]}'
""")

DataFrame[]

In [0]:
spark.sql(f"""
CREATE TABLE {TABLES["gold_product"]} (
    stock_code STRING,
    description STRING,
    units_sold BIGINT,
    product_revenue DOUBLE,
    order_count BIGINT,
    updated_at TIMESTAMP
)
USING DELTA
LOCATION '{PATHS["gold_product"]}'
""")

DataFrame[]

In [0]:
spark.sql(f"""
CREATE TABLE {TABLES["gold_country"]} (
    country STRING,
    year INT,
    month INT,
    total_revenue DOUBLE,
    items_sold BIGINT,
    orders BIGINT,
    updated_at TIMESTAMP
)
USING DELTA
PARTITIONED BY (year, month)
LOCATION '{PATHS["gold_country"]}'
""")

DataFrame[]

In [0]:
spark.sql(f"""
CREATE TABLE {TABLES["gold_customer"]} (
    customer_id DOUBLE,
    total_orders BIGINT,
    total_items BIGINT,
    total_revenue DOUBLE,
    avg_order_value DOUBLE,
    updated_at TIMESTAMP
)
USING DELTA
LOCATION '{PATHS["gold_customer"]}'
""")

DataFrame[]

In [0]:
from config import PATHS

spark.sql(f"""
CREATE TABLE IF NOT EXISTS dbw_retails.monitoring.pipeline_metrics (
    pipeline STRING,
    pipeline_stage STRING,
    row_count BIGINT,
    processed_at TIMESTAMP
)
USING DELTA
LOCATION '{PATHS["monitoring"]}'
""")

DataFrame[]

In [0]:
%sql
ALTER TABLE dbw_retails.bronze.eventhub_transactions
ADD COLUMNS (invoice_ts TIMESTAMP);